In [2]:
import sklearn
import numpy
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import kaggle
from tabulate import tabulate

In [3]:
from pathlib import Path

base = Path("./enron_mail_20150507/maildir/allen-p/_sent_mail")
print(list(base.iterdir())[:10])

[WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/10.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/100.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1000.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1001.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1002.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1003.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1004.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/101.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/102.')]


In [4]:
from pathlib import Path
import shutil
import os

#RAW_ROOT = Path(r"********CURRENT DATA FOLDER*******")
#WIN_ROOT = Path(r"******YOUR NEW FOLDER HERE*******")

def make_windows_safe_name(name: str) -> str:
    """Convert '123.' -> '123.txt'"""
    return name.rstrip(".") + ".txt"

def copy_dataset_safe(raw_root: Path, win_root: Path):
    win_root.mkdir(parents=True, exist_ok=True)

    for src_path in raw_root.iterdir():

        safe_name = make_windows_safe_name(src_path.name)
        dst_file = win_root / safe_name
        
        if dst_file.exists():
            continue
        
        if src_path.is_dir():
            copy_dataset_safe(src_path, win_root / src_path.name)    
        else:
           # Windows-safe name
            dst_file.parent.mkdir(parents=True, exist_ok=True)

            # Use Windows UNC prefix to reliably open files ending with dot
            src_path_unc = Path(r"\\?\\" + str(src_path.resolve()))

            try:
                with open(src_path_unc, "r", encoding="latin-1", errors="ignore") as f_src:
                    content = f_src.read()
                with open(dst_file, "w", encoding="latin-1", errors="ignore") as f_dst:
                    f_dst.write(content)
            except Exception as e:
                print(f"⚠️ Skipped {src_path}: {e}") 
            
# Run copy function to convert dataset from UNIX -> Windows .txt files
#copy_dataset_safe(RAW_ROOT, WIN_ROOT)
print("Windows-safe Enron dataset created successfully.")


Windows-safe Enron dataset created successfully.


In [5]:
import re

def remove_reply_chain(text: str):

    reply_patterns = [
        r"-----Original Message-----",
        r"----- Forwarded by",
        r"\s+From: .*",
        r"\s+Sent: .*",
        r"\s+To: .*",
        r"\s+Subject: .*",
        r"On .* wrote:",
        r"\S+@\S+\s+on\s+\d{1,2}/\d{1,2}/\d{4}" # Looking for email@address on D/M/Y 
    ]

    for pattern in reply_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            text = text[:match.start()]

    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [38]:
from email import policy
from email.parser import Parser
from pathlib import Path

def parse_email(path: Path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        msg = Parser(policy=policy.default).parse(f)
    data = {
        "person": path.parts[1],
        "subfolder": path.parts[2],
        "filename": path.parts[3],
        "message_id": msg["Message-ID"],
        "date": msg["Date"],
        "from": msg["From"],
        "to": msg["To"],
        "cc": msg["Cc"],
        "bcc": msg["Bcc"],
        "subject": msg["Subject"],
        "body": msg.get_body(preferencelist=('plain')).get_content()
            if msg.get_body()
            else msg.get_payload()
    }

    data["body"] = remove_reply_chain(data["body"])
    
    return data

pathtest = Path(r"./enron_windows_dataset/arnold-j/_sent_mail/111.txt")
print(parse_email(pathtest))

{'person': 'arnold-j', 'subfolder': '_sent_mail', 'filename': '111.txt', 'message_id': '<22562856.1075857596586.JavaMail.evans@thyme>', 'date': 'Sat, 14 Oct 2000 10:00:00 -0700', 'from': 'john.arnold@enron.com', 'to': 'slafontaine@globalp.com', 'cc': None, 'bcc': None, 'subject': 'Re: mkts', 'body': "pira's certainly got the whole market wound up. I've seen a wave of producer selling for the first time in two months over the past two days. Most selling Cal 1 off the back of Pira. Pira certainly commands a lot of respect these days. Too much, probably. The problem with all these bull spreads (ie F-H) is the thought process in natty is that if Jan is strong, just think what happens when you get to March and run out of gas. The spread game is very different than playing crude. These spreads haven't moved for the past 1000 point runup. You know there were guys bullish this market trying to play it with spreads and haven't made a penny. Just to clarify, Pira said 3 bcf y on y for Z1? That s

In [31]:
pathtest = Path(r"./enron_windows_dataset/allen-p/_sent_mail")

print(pathtest.parts)

('enron_windows_dataset', 'allen-p', '_sent_mail')


In [36]:
from pathlib import Path
import pandas as pd

def create_pandas_dataset(start_folder: Path):

    dataset = []
    
    for file_path in start_folder.iterdir():

        if file_path.name == ".ipynb_checkpoints":
            continue
        
        if file_path.is_dir():
            data = create_pandas_dataset(file_path)
            dataset.extend(data)
        elif file_path.suffix == ".txt":
            data = parse_email(file_path)
            dataset.append(data)  

    return dataset

def create_dataframe(dataset):

    df = pd.DataFrame(dataset)
    return df

In [39]:
root = Path(r"./enron_windows_dataset/allen-p/_sent_mail")

data = create_pandas_dataset(root)
print(len(data))
df = create_dataframe(data)
print(df.head())

602
    person   subfolder  filename  \
0  allen-p  _sent_mail     1.txt   
1  allen-p  _sent_mail    10.txt   
2  allen-p  _sent_mail   100.txt   
3  allen-p  _sent_mail  1000.txt   
4  allen-p  _sent_mail  1001.txt   

                                      message_id  \
0  <18782981.1075855378110.JavaMail.evans@thyme>   
1  <15464986.1075855378456.JavaMail.evans@thyme>   
2  <24216240.1075855687451.JavaMail.evans@thyme>   
3  <13505866.1075863688222.JavaMail.evans@thyme>   
4  <30922949.1075863688243.JavaMail.evans@thyme>   

                              date                     from  \
0  Mon, 14 May 2001 16:39:00 -0700  phillip.allen@enron.com   
1  Fri, 04 May 2001 13:51:00 -0700  phillip.allen@enron.com   
2  Wed, 18 Oct 2000 03:00:00 -0700  phillip.allen@enron.com   
3  Mon, 23 Oct 2000 06:13:00 -0700  phillip.allen@enron.com   
4  Thu, 31 Aug 2000 05:07:00 -0700  phillip.allen@enron.com   

                        to    cc   bcc    subject  \
0     tim.belden@enron.com  None  

In [40]:
print(df[df['body'] == '-----------------'])

      person   subfolder filename  \
9    allen-p  _sent_mail  102.txt   
12   allen-p  _sent_mail  105.txt   
13   allen-p  _sent_mail  106.txt   
16   allen-p  _sent_mail  109.txt   
18   allen-p  _sent_mail  110.txt   
..       ...         ...      ...   
586  allen-p  _sent_mail   85.txt   
587  allen-p  _sent_mail   86.txt   
588  allen-p  _sent_mail   87.txt   
591  allen-p  _sent_mail    9.txt   
596  allen-p  _sent_mail   94.txt   

                                        message_id  \
9    <30795301.1075855687494.JavaMail.evans@thyme>   
12   <13116875.1075855687561.JavaMail.evans@thyme>   
13    <2707340.1075855687584.JavaMail.evans@thyme>   
16   <19773657.1075855687649.JavaMail.evans@thyme>   
18   <12759088.1075855687671.JavaMail.evans@thyme>   
..                                             ...   
586  <31924171.1075855687124.JavaMail.evans@thyme>   
587  <14186646.1075855687147.JavaMail.evans@thyme>   
588  <30734779.1075855687169.JavaMail.evans@thyme>   
591  <16509612.